In [1]:
# Notebook 3 — Jointures et agrégations

In [2]:
from pyspark.sql import SparkSession
spark =  (SparkSession.builder
            .appName("TradeCorp ETL")
            .getOrCreate()
        )

In [3]:

df_customers = spark.read.parquet("/home/jovyan/data/tmp/customers.parquet")
df_employees = spark.read.parquet("/home/jovyan/data/tmp/employees_enrich.parquet")
df_orders = spark.read.parquet("/home/jovyan/data/tmp/orders.parquet")
df_order_details = spark.read.parquet("/home/jovyan/data/tmp/order_details.parquet")
df_categories = spark.read.parquet("/home/jovyan/data/tmp/categories.parquet")
df_products = spark.read.parquet("/home/jovyan/data/tmp/products.parquet")
df_shippers = spark.read.parquet("/home/jovyan/data/tmp/shippers.parquet")
df_suppliers = spark.read.parquet("/home/jovyan/data/tmp/suppliers.parquet")



In [4]:
# Q21 — Jointure orders + customers
# Joindre df_orders et df_customers sur customer_id. Garder uniquement : order_id, company_name, country,
# order_date, freight
df_customers_orders =  df_orders.join(df_customers, on="customer_id",how="inner").select("order_id","company_name","country","order_date","freight")
df_customers_orders.show(1)

+--------+--------------------+-------+----------+-------+
|order_id|        company_name|country|order_date|freight|
+--------+--------------------+-------+----------+-------+
|   10248|Vins et alcools C...| FRANCE|1996-07-04|  32.38|
+--------+--------------------+-------+----------+-------+
only showing top 1 row


In [5]:
# Q22 — Jointure order_details + products
# Joindre df_order_details et df_products sur product_id. Ajouter les colonnes product_name, category_id,
# unit_price depuis products.
df_order_detailts_products = df_order_details.join(df_products.select("product_name","category_id","unit_price","product_id"), on="product_id",how="inner")
df_order_detailts_products.show(1)

+----------+--------+-------------+--------+--------+----------+--------------+-----------+----------+
|product_id|order_id|prix_unitaire|quantite|discount|sous_total|  product_name|category_id|unit_price|
+----------+--------+-------------+--------+--------+----------+--------------+-----------+----------+
|        11|   10248|         14.0|      12|     0.0|     168.0|Queso Cabrales|          4|      21.0|
+----------+--------+-------------+--------+--------+----------+--------------+-----------+----------+
only showing top 1 row


In [6]:
# Q23 — Jointure products + categories
# Joindre df_products et df_categories sur category_id pour enrichir chaque produit avec category_name et
# description.
df_products_categories = df_products.join(df_categories.select("category_id","category_name","description"), on="category_id", how="inner")
df_products_categories.show(1)

+-----------+----------+------------+-----------+------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|category_id|product_id|product_name|supplier_id| quantity_per_unit|unit_price|units_in_stock|units_on_order|reorder_level|discontinued|en_stock|category_name|         description|
+-----------+----------+------------+-----------+------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|          1|         1|        Chai|          8|10 boxes x 30 bags|      18.0|            39|             0|           10|           1|    true|    Beverages|Soft drinks, coff...|
+-----------+----------+------------+-----------+------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
only showing top 1 row


In [7]:
# Q24 — DataFrame enrichi complet
# A - Réaliser une première jointure complète (order_details, orders, customers, products enrichi avec
# categories, employees, shippers) sans renommer aucune colonne. Lister ensuite les colonnes qui apparaissent
# en double grâce à Counter.

df_orders_enriched = df_orders\
    .join(df_order_details, on="order_id",how="inner")\
    .join(df_customers, on="customer_id",how="inner")\
    .join(df_products_categories, on="product_id", how="inner")\
    .join(df_employees, on="employee_id",how="inner")\
    .join(df_shippers,on="shipper_id",how="inner")


In [14]:
# B - Pour chaque colonne identifiée en Q24a, la renommer dans sa table d'origine avant de refaire la jointure,
# en la préfixant selon la table (customer_country, employee_country, shipper_name...). Reconstruire ensuite
# df_orders_enriched avec ces tables renommées, puis vérifier qu'il ne reste plus aucun doublon

list_col = df_orders_enriched.columns
# df_list_col = spark.createDataFrame(list_col_dup, ["name"])
# df_list_col.groupBy("name").count().show()
 
col_dupli = []
for col in list_col:
    if list_col.count(col)> 1:
        col_dupli.append(col)
print(col_dupli)

df_all = {
    'customers':df_customers,
    'orders':df_orders,
    'product_categories':df_products_categories,
    'order_details':df_order_details,
    'shippers': df_shippers,
    'employees': df_employees
}
for name in df_all:
    df_customers_col = df_all[name].columns
    for col_customers in df_customers_col: 
        if col_customers in col_dupli:
            df_all[name] = df_all[name].withColumnRenamed(col_customers,name+'_'+ col_customers)


df_all["employees"].columns
            



['company_name', 'city', 'country', 'phone', 'city', 'country', 'company_name', 'phone']


['employee_id',
 'first_name',
 'last_name',
 'title',
 'hire_date',
 'employees_city',
 'employees_country',
 'full_name']

In [16]:
df_customers_renamed = df_all["customers"]
df_orders_renamed = df_all["orders"]
df_products_categories_renamed = df_all["product_categories"]
df_order_details_renamed = df_all["order_details"]
df_shippers_renamed = df_all["shippers"]
df_employees_renamed = df_all["employees"]

df_employees_renamed.columns


['employee_id',
 'first_name',
 'last_name',
 'title',
 'hire_date',
 'employees_city',
 'employees_country',
 'full_name']

In [17]:
df_orders_enriched = df_orders_renamed.join(df_order_details_renamed, on="order_id",how="inner")\
    .join(df_customers_renamed, on="customer_id",how="inner")\
    .join(df_products_categories_renamed, on="product_id", how="inner")\
    .join(df_employees_renamed, on="employee_id",how="inner")\
    .join(df_shippers_renamed,on="shipper_id",how="inner")


In [18]:
df_orders_enriched.columns

['shipper_id',
 'employee_id',
 'product_id',
 'customer_id',
 'order_id',
 'order_date',
 'required_date',
 'shipped_date',
 'freight',
 'ship_name',
 'ship_address',
 'ship_city',
 'ship_region',
 'ship_postal_code',
 'ship_country',
 'is_shipped',
 'prix_unitaire',
 'quantite',
 'discount',
 'sous_total',
 'customers_company_name',
 'contact_name',
 'contact_title',
 'address',
 'customers_city',
 'region',
 'postal_code',
 'customers_country',
 'customers_phone',
 'fax',
 'category_id',
 'product_name',
 'supplier_id',
 'quantity_per_unit',
 'unit_price',
 'units_in_stock',
 'units_on_order',
 'reorder_level',
 'discontinued',
 'en_stock',
 'category_name',
 'description',
 'first_name',
 'last_name',
 'title',
 'hire_date',
 'employees_city',
 'employees_country',
 'full_name',
 'shippers_company_name',
 'shippers_phone']

In [ ]:
#  Q25 — CA par client
# Calculer le chiffre d'affaires total par client (company_name) depuis df_orders_enriched. Trier par CA
# décroissant. Afficher le top 10.